In [1]:
!pip install fvcore iopath pytorchvideo


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 11.5 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61396 sha256=f97f19efa640e04b8daf38f583d5f593f58bd01cbd31f9093db5e1e4b142d463
  Stored in directory: /root/.cache/pip/wheels/65/71/95/3b8fde5c65c6e4a806e0867c1651dcc71a1cb2f3430e8f355f
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31528 sha256=025e6027f0a612e280ed55794dadc07209ddd58fd2dff882dd93a368d032face
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
  Created wheel for pytorchv

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import torch
import torchaudio
import torchvision.transforms as transforms
import cv2
import os
import glob
import numpy as np
from torchvision.io import read_video
import torch.nn.functional as F
from PIL import Image

# ========== CONFIG ==========
MODEL_PATH_VIDEO = "/content/drive/MyDrive/COSMOSPosterProject/video_densenet.pth"  # Change this to your actual model path
MODEL_PATH_AUDIO = "/content/drive/MyDrive/COSMOSPosterProject/cnn_lstm_parallel_model75.pt"  # Change this to your actual model path
CREMA_D_AUDIO_PATH = "/content/drive/MyDrive/EMODETECT/CREMA-D/Video_MP4"
CREMA_D_VIDEO_PATH = "/content/drive/MyDrive/EMODETECT/CREMA-D/AudioMP3"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Late Fusion Weights
VIDEO_WEIGHT = 0.5
AUDIO_WEIGHT = 0.5

# Load Haar Cascade for face detection
haar_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")




Mounted at /content/drive


In [14]:
import torch
import torch.nn as nn
from torchvision import models
import cv2
import numpy as np
import glob
from PIL import Image
import torchvision.transforms as transforms
from torchvision.io import read_video

# ========== CONFIG ==========
MODEL_PATH = "/content/drive/MyDrive/COSMOSPosterProject/video_densenet.pth" # video_cnn_model?
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Haar Cascade for face detection
haar_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# ========== LOAD PRETRAINED VIDEO MODEL (DENSENET-201) ==========
class VideoModel(nn.Module):
    def __init__(self):
        super(VideoModel, self).__init__()
        self.model = models.densenet201(pretrained=False)  # Use DenseNet-201

        # Modify first layer to accept 40-channel input
        self.model.features.conv0 = nn.Conv2d(40, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Adjust classifier output to match emotion classes (Assume 5 emotions, change if needed)
        num_features = self.model.classifier.in_features
        self.model.classifier = nn.Linear(num_features, 8)

    def forward(self, x):
        return self.model(x)

# Initialize model
video_model = VideoModel()

# Load checkpoint and rename keys if needed
checkpoint = torch.load(MODEL_PATH, map_location="cpu")

# Fix key mismatch: Change "densenet." → "model."
new_state_dict = {}
for key in checkpoint.keys():
    new_key = key.replace("densenet.", "model.")  # Adjust prefix
    new_state_dict[new_key] = checkpoint[key]

# Load modified state_dict
video_model.load_state_dict(new_state_dict, strict=False)
video_model.to(DEVICE)
video_model.eval()

print("✅ Model loaded successfully!")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-14-256eb184de29>:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the function

✅ Model loaded successfully!


In [17]:
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import librosa
import numpy as np
import glob
import os

# ========== CONFIG ==========
MODEL_PATH = "/content/drive/MyDrive/COSMOSPosterProject/cnn_lstm_parallel_model75.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

'''class AudioModel(nn.Module):
    def __init__(self):
        super(AudioModel, self).__init__()

        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # LSTM Feature Extractor
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=1, bidirectional=True, batch_first=True)

        # Fully Connected Layer
        self.fc = nn.Linear(128 * 2, 8)  # Output: 8 emotions

    def forward(self, x):
        batch_size = x.size(0)

        # Pass through CNN
        x = self.cnn(x)  # Shape: (batch, 64, height, time)
        x = x.permute(0, 3, 2, 1).contiguous()  # Reshape to (batch, time, height, 64)
        x = x.view(batch_size, x.size(1), -1)  # Flatten height dimension

        # Ensure LSTM receives input with shape (batch, time, 64)
        x = x[:, :, :64]  # Trim or pad the last dimension to 64

        # Pass through LSTM
        x, _ = self.lstm(x)

        # Fully Connected Layer
        OUTPUT
        x = self.fc(x[:, -1, :])  # Take the last LSTM output

        return x'''

class AudioModel(nn.Module):
    def __init__(self,num_emotions):
        super().__init__()
        # conv block
        self.conv2Dblock = nn.Sequential(
            # 1. conv block
            nn.Conv2d(in_channels=1,
                       out_channels=16,
                       kernel_size=3,
                       stride=1,
                       padding=1
                      ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(p=0.3),
            # 2. conv block
            nn.Conv2d(in_channels=16,
                       out_channels=32,
                       kernel_size=3,
                       stride=1,
                       padding=1
                      ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4, stride=4),
            nn.Dropout(p=0.3),
            # 3. conv block
            nn.Conv2d(in_channels=32,
                       out_channels=64,
                       kernel_size=3,
                       stride=1,
                       padding=1
                      ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4, stride=4),
            nn.Dropout(p=0.3),
            # 4. conv block
            nn.Conv2d(in_channels=64,
                       out_channels=64,
                       kernel_size=3,
                       stride=1,
                       padding=1
                      ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4, stride=4),
            nn.Dropout(p=0.3)
        )
        # LSTM block
        self.lstm_maxpool = nn.MaxPool2d(kernel_size=[2,4], stride=[2,4])
        hidden_size = 128
        self.lstm = nn.LSTM(input_size=64,hidden_size=hidden_size,bidirectional=True, batch_first=True)
        self.dropout_lstm = nn.Dropout(0.1)
        self.attention_linear = nn.Linear(2*hidden_size,1) # 2*hidden_size for the 2 outputs of bidir LSTM
        # Linear softmax layer
        self.out_linear = nn.Linear(2*hidden_size+256,num_emotions)
        self.dropout_linear = nn.Dropout(p=0)
        self.out_softmax = nn.Softmax(dim=1)
    def forward(self,x):
        # conv embedding
        conv_embedding = self.conv2Dblock(x) #(b,channel,freq,time)
        conv_embedding = torch.flatten(conv_embedding, start_dim=1) # do not flatten batch dimension
        # lstm embedding
        x_reduced = self.lstm_maxpool(x)
        x_reduced = torch.squeeze(x_reduced,1)
        x_reduced = x_reduced.permute(0,2,1) # (b,t,freq)
        lstm_embedding, (h,c) = self.lstm(x_reduced) # (b, time, hidden_size*2)
        lstm_embedding = self.dropout_lstm(lstm_embedding)
        batch_size,T,_ = lstm_embedding.shape
        attention_weights = [None]*T
        for t in range(T):
            embedding = lstm_embedding[:,t,:]
            attention_weights[t] = self.attention_linear(embedding)
        attention_weights_norm = nn.functional.softmax(torch.stack(attention_weights,-1),-1)
        attention = torch.bmm(attention_weights_norm,lstm_embedding) # (Bx1xT)*(B,T,hidden_size*2)=(B,1,2*hidden_size)
        attention = torch.squeeze(attention, 1)
        # concatenate
        complete_embedding = torch.cat([conv_embedding, attention], dim=1)

        output_logits = self.out_linear(complete_embedding)
        output_logits = self.dropout_linear(output_logits)
        output_softmax = self.out_softmax(output_logits)
        '''OUTPUT'''
        return output_logits, output_softmax, attention_weights_norm

# Initialize model
audio_model = AudioModel(8)

# Load model weights
checkpoint = torch.load(MODEL_PATH, map_location="cpu")
audio_model.load_state_dict(checkpoint, strict=False)
audio_model.to(DEVICE)
audio_model.eval()

print("✅ Audio Model loaded successfully!")

✅ Audio Model loaded successfully!


<ipython-input-17-2f8b0caec317>:151: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_location="cpu")


In [18]:
# ========== ADD WHITE GAUSSIAN NOISE (WGN) ==========
def add_white_gaussian_noise(audio, snr_db):
    """Add White Gaussian Noise to an audio signal given an SNR level in dB."""
    rms_signal = np.sqrt(np.mean(audio**2))
    rms_noise = rms_signal / (10**(snr_db / 20))
    noise = np.random.normal(0, rms_noise, audio.shape)
    return audio + noise

# ========== MEL SPECTROGRAM CONVERSION ==========
def process_audio(audio_path):
    """Load an audio file, apply noise augmentation, and convert it into Mel Spectrograms."""
    y, sr = librosa.load(audio_path, sr=16000)  # Load at 16kHz

    # Create three versions: Original, SNR 15, SNR 30
    augmented_signals = [y, add_white_gaussian_noise(y, 15), add_white_gaussian_noise(y, 30)]
    spectrograms = []

    for signal in augmented_signals:
        # Convert to Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128, fmax=8000, hop_length=256, win_length=512, window='hamming')
        mel_spec = librosa.power_to_db(mel_spec, ref=np.max)  # Convert to dB
        mel_spec = torch.tensor(mel_spec).unsqueeze(0)  # Shape: (1, 128, Time)
        spectrograms.append(mel_spec)

    return torch.stack(spectrograms).to(DEVICE)  # Shape: (3, 1, 128, Time)

In [19]:
# ========== VIDEO PREPROCESSING FUNCTION ==========
def process_video(video_path):
    frames, _, _ = read_video(video_path, pts_unit="sec")

    processed_frames = []
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
        transforms.Resize((48, 48)),  # Resize to 48x48
        transforms.ToTensor(),  # Convert to tensor
        transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize
    ])

    for i in range(min(40, len(frames))):  # Process first 40 frames
        frame = frames[i].numpy()
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)  # Convert to grayscale

        # Detect face using Haar Cascade
        faces = haar_cascade.detectMultiScale(frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        if len(faces) > 0:
            x, y, w, h = faces[0]  # Take the first detected face
            frame = frame[y:y+h, x:x+w]  # Crop the face

        # Convert to PIL Image and apply transformations
        frame_pil = Image.fromarray(frame)
        frame_tensor = transform(frame_pil)
        processed_frames.append(frame_tensor)

    if len(processed_frames) == 0:
        return None  # Skip this sample if no face detected

    # Stack frames into a (40, 1, 48, 48) tensor
    video_tensor = torch.stack(processed_frames).squeeze(1).unsqueeze(0)  # Shape: (1, 40, 48, 48)
    return video_tensor.to(DEVICE)


In [20]:
import os
import glob
import torch
import torch.nn.functional as F

drive.mount('/content/drive', force_remount=True)


# Paths to Video and Audio Data
VIDEO_PATH = "/content/drive/MyDrive/EMODETECT/CREMA-D/Video_MP4"
AUDIO_PATH = "/content/drive/MyDrive/EMODETECT/CREMA-D/AudioWAV"

# Get video files and sort numerically based on filename
video_files = sorted(glob.glob(VIDEO_PATH + "/*.mp4"), key=lambda x: int(os.path.basename(x).split('_')[0]))

# Get audio files and sort numerically based on filename
audio_files = sorted(glob.glob(AUDIO_PATH + "/*.wav"), key=lambda x: int(os.path.basename(x).split('_')[0]))

# Convert file lists to dictionaries for easy lookup
video_dict = {os.path.basename(f).replace(".mp4", ""): f for f in video_files}
audio_dict = {os.path.basename(f).replace(".wav", ""): f for f in audio_files}

print(f"🎥 Found {len(video_files)} MP4 videos.")
print(f"🎤 Found {len(audio_files)} WAV audios.")
print(f"📌 Matching files will be processed, unmatched ones will be skipped.")


Mounted at /content/drive
🎥 Found 7441 MP4 videos.
🎤 Found 7442 WAV audios.
📌 Matching files will be processed, unmatched ones will be skipped.


In [21]:
import os
import torch
import glob
import numpy as np
import subprocess


# Define directories for processed files
PROCESSED_VIDEO_PATH = "/content/drive/MyDrive/EMODETECT/Processed_Video"
PROCESSED_AUDIO_PATH = "/content/drive/MyDrive/EMODETECT/Processed_Audio"

# Ensure the directories exist
os.makedirs(PROCESSED_VIDEO_PATH, exist_ok=True)
os.makedirs(PROCESSED_AUDIO_PATH, exist_ok=True)

print("🚀 Starting Preprocessing of Video & Audio files...")

# Function to add Gaussian noise
def add_white_gaussian_noise(audio, snr_db):
    """Add White Gaussian Noise to an audio signal given an SNR level in dB."""
    rms_signal = np.sqrt(np.mean(audio**2))
    rms_noise = rms_signal / (10**(snr_db / 20))
    noise = np.random.normal(0, rms_noise, audio.shape)
    return audio + noise

# Iterate through sorted video files
for base_filename in sorted(video_dict.keys()):  # Process in numerical order
    video_path = video_dict[base_filename]
    audio_path = audio_dict.get(base_filename, None)

    # Define processed file paths
    processed_video_file = os.path.join(PROCESSED_VIDEO_PATH, f"{base_filename}.pt")
    processed_audio_file = os.path.join(PROCESSED_AUDIO_PATH, f"{base_filename}_clean.pt")
    processed_audio_snr15_file = os.path.join(PROCESSED_AUDIO_PATH, f"{base_filename}_snr15.pt")
    processed_audio_snr30_file = os.path.join(PROCESSED_AUDIO_PATH, f"{base_filename}_snr30.pt")

    # Preprocess & Save Video
    if os.path.exists(processed_video_file):
        print(f"📂 Skipping {video_path}, already processed.")

    else:
      try:
        video_input = process_video(video_path)
        if video_input is not None:
            torch.save(video_input.cpu(), processed_video_file)
            print(f"💾 Saved processed video: {processed_video_file}")
      except:
        print("stop")

    # Preprocess & Save Audio (Original + Noisy Versions)
    if audio_path:
        if os.path.exists(processed_audio_file) and os.path.exists(processed_audio_snr15_file) and os.path.exists(processed_audio_snr30_file):
            print(f"📂 Skipping {audio_path}, already processed.")
        else:
            audio_signal = process_audio(audio_path)
            if audio_signal is not None:
                # Convert to NumPy for noise augmentation
                audio_numpy = audio_signal.cpu().numpy()

                # Generate noisy versions
                audio_snr15 = add_white_gaussian_noise(audio_numpy, 15)
                audio_snr30 = add_white_gaussian_noise(audio_numpy, 30)

                # Convert back to tensors
                torch.save(torch.tensor(audio_numpy), processed_audio_file)
                torch.save(torch.tensor(audio_snr15), processed_audio_snr15_file)
                torch.save(torch.tensor(audio_snr30), processed_audio_snr30_file)

                print(f"💾 Saved processed audio: {processed_audio_file}, {processed_audio_snr15_file}, {processed_audio_snr30_file}")

    else:
        print(f"⚠️ No matching audio for {video_path}, skipping audio processing.")

print("✅ Preprocessing complete. All processed files are cached!")


Streaming output truncated to the last 5000 lines.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/AudioWAV/1061_IWL_DIS_XX.wav, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/Video_MP4/1061_IWL_FEA_XX.mp4, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/AudioWAV/1061_IWL_FEA_XX.wav, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/Video_MP4/1061_IWL_HAP_XX.mp4, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/AudioWAV/1061_IWL_HAP_XX.wav, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/Video_MP4/1061_IWL_NEU_XX.mp4, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/AudioWAV/1061_IWL_NEU_XX.wav, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/Video_MP4/1061_IWL_SAD_XX.mp4, already processed.
📂 Skipping /content/drive/MyDrive/EMODETECT/CREMA-D/AudioWAV/1061_IWL_SAD_XX.wav, already processed.
📂 Skipping /content/drive/MyDrive/EM

In [22]:
import torch

# Load a sample preprocessed video and audio tensor
sample_video = torch.load("/content/drive/MyDrive/EMODETECT/Processed_Video/1001_DFA_ANG_XX.pt")
sample_audio = torch.load("/content/drive/MyDrive/EMODETECT/Processed_Audio/1001_DFA_ANG_XX_clean.pt")

print("Video Shape:", sample_video.shape)  # Expected: (1, 40, 1, 48, 48)
print("Audio Shape:", sample_audio.shape)  # Expected: (1, time_steps, 64)


Video Shape: torch.Size([1, 40, 48, 48])
Audio Shape: torch.Size([3, 1, 128, 142])


<ipython-input-22-2a07ce0493e2>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sample_video = torch.load("/content/drive/MyDrive/EMODETECT/Processed_Video/1001_DFA_ANG_XX.

In [23]:
import os
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Define the five target emotions from filenames
EMOTION_MAPPING = {
    "ANG": "angry",
    "CAL": "calm",
    "DIS": "disgust",
    "NEU": "neutral",
    "SAD": "sad",
    "FEA": "fear",
    "HAP": "happy"
}

# Model emotion labels (Ensure these match your model output order)
MODEL_EMOTIONS = ["happy", "angry", "neutral", "sad", "disgust", "fear", "surprise", "calm"]

# Get indices of the five target emotions (ignore emotions not in TARGET_EMOTIONS)
TARGET_EMOTIONS = ["calm", "angry", "neutral", "sad", "happy"]
TARGET_INDICES = [MODEL_EMOTIONS.index(emotion) for emotion in TARGET_EMOTIONS if emotion in MODEL_EMOTIONS]

# Define Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move models to DEVICE
video_model.to(DEVICE)
audio_model.to(DEVICE)

# Directories for preprocessed files
PROCESSED_VIDEO_PATH = "/content/drive/MyDrive/EMODETECT/Processed_Video"
PROCESSED_AUDIO_PATH = "/content/drive/MyDrive/EMODETECT/Processed_Audio"

# Set a limit on the number of files to process (Change as needed)
FILE_LIMIT = 1000  # Modify this to process more or fewer files

# Dictionary to store results
true_labels = []
pred_labels_fusion = []
pred_labels_video = []
pred_labels_audio = []

print(f"🚀 Running predictions using preprocessed video & audio files (Processing up to {FILE_LIMIT} files)...")

# Get a list of preprocessed files, limited by FILE_LIMIT
file_list = sorted(os.listdir(PROCESSED_VIDEO_PATH))[:FILE_LIMIT]

for base_filename in file_list:
    base_filename = base_filename.replace(".pt", "")

    processed_video_file = os.path.join(PROCESSED_VIDEO_PATH, f"{base_filename}.pt")
    processed_audio_file = os.path.join(PROCESSED_AUDIO_PATH, f"{base_filename}_clean.pt")

    # Ensure both processed video & audio exist
    if not (os.path.exists(processed_video_file) and os.path.exists(processed_audio_file)):
        print(f"⚠️ Skipping {base_filename}, preprocessed data missing.")
        continue

    # Extract ground truth from filename
    emotion_code = base_filename.split("_")[2]  # Example: "1001_IEO_ANG_HI" → "ANG"
    ground_truth = EMOTION_MAPPING.get(emotion_code, None)

    if ground_truth is None or ground_truth not in TARGET_EMOTIONS:
        print(f"⚠️ Skipping {base_filename}, ground truth emotion not in target emotions.")
        continue

    try:
        # **Load preprocessed tensors and move to DEVICE**
        video_input = torch.load(processed_video_file, map_location="cpu").float().to(DEVICE)
        audio_input = torch.load(processed_audio_file, map_location="cpu").float().to(DEVICE)

        # **Fix Video Shape Issue**
        video_input = video_input.squeeze(1)  # Remove extra channel dim

        if len(video_input.shape) == 3:
            video_input = video_input.unsqueeze(0)  # Ensure shape [1, 40, 48, 48]

        # **Fix Audio Shape Issue**
        if len(audio_input.shape) == 3:
            audio_input = audio_input.unsqueeze(1)  # Ensure shape [batch_size, 1, freq, time]

        # **Handle multiple audio versions (clean, SNR 15, SNR 30)**
        if len(audio_input.shape) == 4 and audio_input.shape[0] > 1:
            audio_input = torch.mean(audio_input, dim=0, keepdim=True)  # Take mean over augmented versions

        print(f"✅ Running inference on {base_filename}...")

        with torch.no_grad():
            # Run both models
            video_output = video_model(video_input)
            audio_output = audio_model(audio_input)

            # Apply softmax for probability distribution
            video_probs = F.softmax(video_output, dim=1)
            audio_probs = F.softmax(audio_output, dim=1)

            # **Late fusion (average audio versions before fusion)**
            final_probs = (video_probs + audio_probs) / 2

            # **Filter to keep only the 5 target emotions**
            filtered_probs_fusion = final_probs[:, TARGET_INDICES]
            filtered_probs_video = video_probs[:, TARGET_INDICES]
            filtered_probs_audio = audio_probs[:, TARGET_INDICES]

            # **Get predictions**
            prediction_idx_fusion = torch.argmax(filtered_probs_fusion, dim=1).item()
            prediction_idx_video = torch.argmax(filtered_probs_video, dim=1).item()
            prediction_idx_audio = torch.argmax(filtered_probs_audio, dim=1).item()

        final_emotion_fusion = TARGET_EMOTIONS[prediction_idx_fusion]
        final_emotion_video = TARGET_EMOTIONS[prediction_idx_video]
        final_emotion_audio = TARGET_EMOTIONS[prediction_idx_audio]

        print(f"🎬 File: {base_filename} - Fusion: {final_emotion_fusion} | Video: {final_emotion_video} | Audio: {final_emotion_audio} | Ground Truth: {ground_truth}")

        # Store for accuracy computation
        true_labels.append(ground_truth)
        pred_labels_fusion.append(final_emotion_fusion)
        pred_labels_video.append(final_emotion_video)
        pred_labels_audio.append(final_emotion_audio)

    except Exception as e:
        print(f"⚠️ Error processing {base_filename}: {e}")


# Compute Accuracy and Metrics
accuracy_fusion = accuracy_score(true_labels, pred_labels_fusion)
accuracy_video = accuracy_score(true_labels, pred_labels_video)
accuracy_audio = accuracy_score(true_labels, pred_labels_audio)

precision_fusion, recall_fusion, f1_fusion, _ = precision_recall_fscore_support(true_labels, pred_labels_fusion, average="weighted")
precision_video, recall_video, f1_video, _ = precision_recall_fscore_support(true_labels, pred_labels_video, average="weighted")
precision_audio, recall_audio, f1_audio, _ = precision_recall_fscore_support(true_labels, pred_labels_audio, average="weighted")

print("\n🔍 **Final Evaluation Metrics:**")
print(f"✅ Overall Accuracy (Fusion Model): {accuracy_fusion:.4f}")
print(f"✅ Video-Only Model Accuracy: {accuracy_video:.4f}")
print(f"✅ Audio-Only Model Accuracy: {accuracy_audio:.4f}")

print("\n🔍 **Precision, Recall, F1-Scores:**")
print(f"🎥 Video Model - Precision: {precision_video:.4f}, Recall: {recall_video:.4f}, F1-Score: {f1_video:.4f}")
print(f"🎤 Audio Model - Precision: {precision_audio:.4f}, Recall: {recall_audio:.4f}, F1-Score: {f1_audio:.4f}")
print(f"🤝 Fusion Model - Precision: {precision_fusion:.4f}, Recall: {recall_fusion:.4f}, F1-Score: {f1_fusion:.4f}")


🚀 Running predictions using preprocessed video & audio files (Processing up to 1000 files)...
✅ Running inference on 1001_DFA_ANG_XX...


<ipython-input-23-13ef5d28950e>:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  video_input = torch.load(processed_video_file, map_location="cpu").float().to(DEVICE)
<ipyt

⚠️ Error processing 1001_DFA_ANG_XX: mat1 and mat2 shapes cannot be multiplied (1x320 and 512x8)
✅ Running inference on 1001_DFA_DIS_XX...
⚠️ Error processing 1001_DFA_DIS_XX: mat1 and mat2 shapes cannot be multiplied (1x320 and 512x8)
⚠️ Skipping 1001_DFA_FEA_XX, ground truth emotion not in target emotions.
⚠️ Skipping 1001_DFA_HAP_XX, ground truth emotion not in target emotions.
✅ Running inference on 1001_DFA_NEU_XX...
⚠️ Error processing 1001_DFA_NEU_XX: mat1 and mat2 shapes cannot be multiplied (1x320 and 512x8)
✅ Running inference on 1001_DFA_SAD_XX...
⚠️ Error processing 1001_DFA_SAD_XX: Given input size: (64x4x3). Calculated output size: (64x1x0). Output size is too small
✅ Running inference on 1001_IEO_ANG_HI...
⚠️ Error processing 1001_IEO_ANG_HI: Given input size: (64x4x3). Calculated output size: (64x1x0). Output size is too small
✅ Running inference on 1001_IEO_ANG_LO...
⚠️ Error processing 1001_IEO_ANG_LO: mat1 and mat2 shapes cannot be multiplied (1x320 and 512x8)
✅ Runn

KeyboardInterrupt: 